In [ ]:
from dataclasses import dataclass
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_mistralai import ChatMistralAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command

load_dotenv()

True

# Short Term Memory (State)

In [79]:
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """ Get the most recent message by the user """
    messages= runtime.state.get('messages', [])
    
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content
        
    return "No messages found"

@tool
def get_user_pref(pref_name: str, runtime: ToolRuntime) -> str:
    """ Get the user's preference """
    preferences= runtime.state.get('preferences', {})
    return preferences.get(pref_name, "Not Set")

@tool
def set_user_name(new_name: str, runtime: ToolRuntime) -> Command:
    """Set the user's name in the conversation state."""

    tool_call_id = runtime.tool_call_id

    return Command(
        update={
            "user_name": new_name,
            "messages": [ToolMessage(content=f"User name set to {new_name}", tool_call_id=tool_call_id)]
        }
    )

In [80]:
model= ChatMistralAI(model='mistral-medium-latest')

agent= create_agent(
    model= model,
    tools= [get_last_user_message, get_user_pref, set_user_name],
    checkpointer= InMemorySaver(),
    system_prompt='You are a helpful agent. You have access to certain tools that helps you return details from chat history.'
)

In [94]:
query= HumanMessage(content='What did I say in my last message?')
config= {'configurable': {'thread_id': '1'}}
response= agent.invoke({"messages": query}, config= config)
response

{'messages': [HumanMessage(content='What is your name?', additional_kwargs={}, response_metadata={}, id='19b64497-96f2-4077-8db6-feab5f875afc'),
  AIMessage(content="I don't have a name. But I can set a name for you! Would you like me to call you by a specific name?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 202, 'total_tokens': 231, 'completion_tokens': 29, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cdbe1-a1c1-7b61-ad51-17a9678ddd35-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 29, 'total_tokens': 231}),
  HumanMessage(content='How is your day going?', additional_kwargs={}, response_metadata={}, id='8f87ab28-e1d4-4009-9703-261b618919d7'),
  AIMessage(content="I don't have days like humans do, but I'm here and ready to help you! How about you? How's your da

# Context

In [ ]:
USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

In [102]:
@dataclass
class UserContext:
    user_id: str
    
@tool
def get_user_info(runtime: ToolRuntime[UserContext]) -> str:
    """ Get the current user's information """
    user_id= runtime.context.user_id
    
    if user_id in USER_DATABASE:
        user= USER_DATABASE[user_id]
        return f"Account Owner: {user['name']}\nAccount Type: {user['account_type']}\nBalance: {user['balance']}\nEmail: {user['email']}"
    return "User does not exist in database"

In [103]:
model= ChatMistralAI(model='mistral-medium-latest')

agent= create_agent(
    model= model,
    tools= [get_user_info],
    context_schema= UserContext,
    system_prompt='You are a helpful financial assistant. You have access to certain tools, use them whenecer necessary.'
)

In [105]:
result= agent.invoke({
    'messages': HumanMessage(content='What is my balance?')},
    context=UserContext(user_id= 'user123')                     
)

In [106]:
result

{'messages': [HumanMessage(content='What is my balance?', additional_kwargs={}, response_metadata={}, id='e86a9d05-15cc-4f39-a42d-b06dbb1df12d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qcYclSyPN', 'function': {'name': 'get_user_info', 'arguments': '{}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 75, 'total_tokens': 82, 'completion_tokens': 7, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cdbf5-17bc-7a80-8594-506aa3a51e2a-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'qcYclSyPN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 7, 'total_tokens': 82}),
  ToolMessage(content='Account Owner: Alice Johnson\nAccount Type: Premium\nBalance: 5000\nEmail: alice@example.com', name='get_user_info', id='a4f11aad-9efb-4df4-af46-fa7b2

# Long Term Memory (Store)

In [129]:
@dataclass
class UserContext:
    user_id: str

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """ Look at the user information """
    user_info= None
    store= runtime.store
    if user_id == runtime.context.user_id:
        user_info = store.get(("users"), user_id)
    else:
        return "You are not authorized to view this user's information"
    
    return str(user_info) if user_info else "User not found"

@tool 
def save_user_tool(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """ Save the user information """
    
    store= runtime.store
    store.put(("users"), user_id, user_info)
    return "Successfully saved user information"



In [130]:
store= InMemoryStore()

model= ChatMistralAI(model='mistral-medium-latest')

agent= create_agent(
    model= model,
    tools= [get_user_info, save_user_tool],
    checkpointer= InMemorySaver(),
    context_schema= UserContext,
    store= store
)

In [137]:
Query= HumanMessage(content="Can you get the information for user def123?")
config= {'configurable': {'thread_id': '1'}}
response= agent.invoke({"messages": Query}, config= config, context= UserContext(user_id= 'def123'))
response

d:\AppStoneLab\GenAI\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='def123'), input_type=UserContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev', additional_kwargs={}, response_metadata={}, id='e0d6bee4-55a9-4a27-8032-f060b42f7d1e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ETG46oGJ3', 'function': {'name': 'save_user_tool', 'arguments': '{"user_id": "abc123", "user_info": {"name": "Foo", "age": 25, "email": "foo@langchain.dev"}}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 171, 'total_tokens': 215, 'completion_tokens': 44, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cdc50-5708-72b3-8e97-6383b9382a3b-0', tool_calls=[{'name': 'save_user_tool', 'args': {'user_id': 'abc123', 'user_info': {'name': 'Foo', 'age': 25, 'email': 'foo@langchain.dev'}}, 'id': 'ETG46oGJ3', 'type': 'tool_call'}], invalid_tool_calls=[], 

# Stream Writer

In [ ]:
@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """ Get the weather in a city """
    writer = runtime.stream_writer

    writer(f"Looking up data for city: {city}")
    

    writer(f"Acquired data for city: {city}")
    

    writer("Processing weather info...")
    

    return f"The weather in {city} is sunny, 25°C."

In [164]:
model= ChatMistralAI(model='mistral-medium-latest')

agent= create_agent(
    model= model,
    tools= [get_weather],
    system_prompt='You are a helpful weather agent. You have access to certain tools, use them whenecer necessary'
)

In [145]:
query= HumanMessage(content='What is the weather in New York?')
response= agent.invoke({"messages": query})
response

{'messages': [HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='9e05492d-5b4c-400f-9a77-6ef563d31a5a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'V5jW3BGh5', 'function': {'name': 'get_weather', 'arguments': '{"city": "New York"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 103, 'total_tokens': 116, 'completion_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cdc5f-ab79-7fc3-a8a5-78d69f61bdc5-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'V5jW3BGh5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 103, 'output_tokens': 13, 'total_tokens': 116}),
  ToolMessage(content='{"city": "New York", "weather": "sunny", "temperature": "25°C"}', name='get_weather', id='4b8673a8-

In [165]:
for event in agent.stream({"messages": [query]}, stream_mode="updates"):
    if "tools" in event:
        for msg in event["tools"]["messages"]:
            print(msg.content)

The weather in New York is sunny, 25°C.
